# Clean the Network file
Preprocess `cross_references.tsv` according to the initial EDA.

In [67]:
import pandas as pd

In [68]:
raw_data = pd.read_csv('data/original_data/cross_references.tsv', sep='\t')
raw_data.sample(10)

,From Verse,To Verse,Votes,#www.openbible.info CC-BY 2026-09-07
297649,1Cor.6.10,1Thess.4.6,6,NaN
21813,Exod.33.11,Isa.42.8,4,NaN
67381,2Sam.24.20,2Sam.9.8,2,NaN
292955,Rom.10.7,Rom.4.25,2,NaN
60655,1Sam.22.18,1Sam.2.28,2,NaN
286052,Acts.23.11,Isa.41.14,4,NaN
111895,Ps.18.7,Ps.68.7-Ps.68.8,4,NaN
6282,Gen.24.42,Ps.37.5,3,NaN
141790,Prov.7.24,1John.2.1,3,NaN
160698,Isa.25.12,Jer.51.58,2,NaN


In [69]:
data = raw_data.iloc[:,0:3]
display(data.shape)
display(data.sample(5))

(344756, 3)

,From Verse,To Verse,Votes
214180,Hos.5.15,Prov.8.17,2
321646,2Tim.1.16,Phlm.1.7,6
142927,Prov.11.20,Prov.9.7,3
207342,Ezek.43.22,1Pet.1.19,2
37444,Num.33.1,1Sam.12.8,2


In [70]:
# remove negative and zero votes
data_positive = data.loc[data['Votes']>0]
display(data_positive.shape)
display(data_positive.sample(5))

(341242, 3)

,From Verse,To Verse,Votes
304119,2Cor.8.21,Titus.2.5-Titus.2.8,6
49211,Josh.10.41,Judg.16.21,2
117897,Ps.42.9,Ps.78.35,2
179835,Jer.15.6,Ps.78.38-Ps.78.40,2
248435,Mark.12.10,Ps.118.22-Ps.118.23,36


In [71]:
def filter_func(row, max_verse):
    # identify if the from verse and to verse are from the same chapter
    chap1 = '.'.join(row['From Verse'].split('.')[:2])
    chap2 = '.'.join(row['To Verse'].split('.')[:2])
    same_chapter = chap1 == chap2

    # if the To Verse spans multiple verses:
    if '-' in row['To Verse']:
        # Check whether it spans multiple chapters
        start, end = row['To Verse'].split('-')
        start_parts = start.split('.')
        end_parts = end.split('.')

        # check whether 'To Verse' stays within one chapter
        not_multi_chapter = start_parts[:2] == end_parts[:2]
        
        # check the number of verses it spans
        start_verse = int(start_parts[2])
        end_verse = int(end_parts[2])
        verse_span = end_verse - start_verse + 1
    else:
        not_multi_chapter = True
        verse_span = 1
    return (not same_chapter) and not_multi_chapter and (verse_span <= max_verse)

In [76]:
# standardizing bible verse format
BOOK_MAP = {
    "Gen": "Genesis",
    "Exod": "Exodus",
    "Lev": "Leviticus",
    "Num": "Numbers",
    "Deut": "Deuteronomy",
    "Josh": "Joshua",
    "Judg": "Judges",
    "Ruth": "Ruth",
    "1Sam": "1 Samuel",
    "2Sam": "2 Samuel",
    "1Kgs": "1 Kings",
    "2Kgs": "2 Kings",
    "1Chr": "1 Chronicles",
    "2Chr": "2 Chronicles",
    "Ezra": "Ezra",
    "Neh": "Nehemiah",
    "Esth": "Esther",
    "Job": "Job",
    "Ps": "Psalm",
    "Prov": "Proverbs",
    "Eccl": "Ecclesiastes",
    "Song": "Song of Solomon",
    "Isa": "Isaiah",
    "Jer": "Jeremiah",
    "Lam": "Lamentations",
    "Ezek": "Ezekiel",
    "Dan": "Daniel",
    "Hos": "Hosea",
    "Joel": "Joel",
    "Amos": "Amos",
    "Obad": "Obadiah",
    "Jonah": "Jonah",
    "Mic": "Micah",
    "Nah": "Nahum",
    "Hab": "Habakkuk",
    "Zeph": "Zephaniah",
    "Hag": "Haggai",
    "Zech": "Zechariah",
    "Mal": "Malachi",

    "Matt": "Matthew",
    "Mark": "Mark",
    "Luke": "Luke",
    "John": "John",
    "Acts": "Acts",
    "Rom": "Romans",
    "1Cor": "1 Corinthians",
    "2Cor": "2 Corinthians",
    "Gal": "Galatians",
    "Eph": "Ephesians",
    "Phil": "Philippians",
    "Col": "Colossians",
    "1Thess": "1 Thessalonians",
    "2Thess": "2 Thessalonians",
    "1Tim": "1 Timothy",
    "2Tim": "2 Timothy",
    "Titus": "Titus",
    "Phlm": "Philemon",
    "Heb": "Hebrews",
    "Jas": "James",
    "1Pet": "1 Peter",
    "2Pet": "2 Peter",
    "1John": "1 John",
    "2John": "2 John",
    "3John": "3 John",
    "Jude": "Jude",
    "Rev": "Revelation",
}


def format_verse(ref):
    parts = ref.split("-")
    first = parts[0].split(".")
    result = f"{BOOK_MAP[first[0]]} {first[1]}:{first[2]}"

    if len(parts) == 2:
        last = parts[1].split(".")
        result += f"-{last[2]}"

    return result

In [77]:
# Apply filter
idx_filter = data_positive.apply(filter_func, max_verse=5, axis=1) # set max verse span to 5
data_filtered = data_positive.loc[idx_filter, :]
data_filtered.loc[:,['From Verse', 'To Verse']] = data_filtered.loc[:,['From Verse', 'To Verse']].map(lambda bible_verse: format_verse(bible_verse))
display(data_filtered.shape)
display(data_filtered.sample(10))

(312791, 3)

,From Verse,To Verse,Votes
292333,Romans 8:39,2 Thessalonians 2:9-12,3
249412,Mark 14:22,Mark 6:41,2
235309,Matthew 11:20,2 Timothy 2:25-26,6
179209,Jeremiah 13:16,Amos 8:9-10,5
230427,Matthew 1:21,Zechariah 9:9,18
292102,Romans 8:30,Ephesians 4:4,7
316824,1 Thessalonians 3:5,2 Corinthians 11:2-3,2
246937,Mark 8:38,Jude 1:14-15,2
171273,Isaiah 56:12,Titus 1:7,3
336809,1 John 4:5,Revelation 12:9,7


In [78]:
# save filtered data
data_filtered.to_csv('data/data_filtered.tsv', sep='\t')

In [79]:
# drop votes column
data_filtered_no_votes = data_filtered.drop(columns=["Votes"])
data_filtered_no_votes.to_csv('data/data_filtered_no_votes.tsv', sep='\t')

# Clean the Bible file

In [20]:
bible_text_path = './data/original_data/kjv.txt'
bible = pd.read_csv(bible_text_path, sep='\t', skiprows=2, header=None)
bible.columns = ['verse', 'text']

bible_csv_path = './data/kjv.tsv'
bible.to_csv(bible_csv_path, sep='\t')